In [32]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [33]:
data = pd.DataFrame() 
years = ["2022"]

for y in years:
    PATH = f"../data/processed/statcast_{y}.parquet"
    data = pd.concat([data, pd.read_parquet(PATH)])

# Keep only umpire ball/strike calls
data = data[data['description'].isin(['called_strike', 'ball'])]

print(data.shape)

(380559, 121)


In [34]:
#called pitches only 
df = data[data['description'].isin(['called_strike', 'ball'])].copy()

In [35]:
x = -0.83
y = 0.83
df = df.dropna(subset=['plate_x', 'plate_z', 'sz_bot', 'sz_top'])

df['expected'] = (
    df['plate_x'].between(x,y) &
    df['plate_z'].between(df['sz_bot'], df['sz_top'])
).astype(int)

In [36]:
# if called strike 1 if ball 0 
df['outcome'] = (
    df['description'] == 'called_strike'
).astype(int)

In [37]:
# do expected & call match 
df['target'] = (
    df['outcome'] != df['expected']
).astype(int)

In [38]:
f_data = [
    'plate_x',
    'plate_z',
    'sz_top',
    'sz_bot',
    'release_speed',
    'balls',
    'strikes',
    'pitch_type',
    'stand',
    'p_throws'
]

target_col = 'target'
cat_col = ['pitch_type', 'stand','p_throws']
#categorical variables
num_col = [col for col in f_data if col not in cat_col] #numeric variables

In [39]:
X_data = df[f_data].copy()
Y_data = df[target_col].copy()

In [40]:
X_data.head()

,plate_x,plate_z,sz_top,sz_bot,release_speed,balls,strikes,pitch_type,stand,p_throws
4,0.652451,1.724418,3.720355,1.723947,94.5,1,0,FF,L,R
5,-0.299552,3.580697,3.658373,1.728693,95.6,0,0,FF,L,R
6,-1.727309,2.289997,3.541271,1.637857,94.4,3,0,FF,R,R
7,0.860265,3.49821,3.529266,1.683829,95.5,2,0,FF,R,R
13,2.652251,0.947512,3.22007,1.642547,85.9,0,1,ST,L,R


In [41]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_cat = encoder.fit_transform(X_data[cat_col])

encoded_df = pd.DataFrame(
    encoded_cat,
    columns=encoder.get_feature_names_out(cat_col),
    index=X_data.index
)

X_data_transformed = pd.concat(
    [X_data[num_col], encoded_df],
    axis=1
)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X_data_transformed,
    Y_data,
    test_size=0.2,
    random_state=42,
    stratify=Y_data
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (297003, 26)
Testing set: (74251, 26)


In [43]:
lr_model = LogisticRegression(max_iter=1000,class_weight='balanced',
    random_state=42) # needed to add in balanced, we have a class imbalance

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values